In [ ]:

from dotenv import load_dotenv
load_dotenv()

from typing_extensions import TypedDict
from typing import Annotated, List
import operator
import os

from langgraph.graph import StateGraph, START, END
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langchain_community.utilities import SQLDatabase
from pydantic import BaseModel, Field

from langchain.agents import create_agent

from scripts import my_tools, mysql_tools
from scripts.nodes import (
    retrieve_node,
    grade_documents_node,
    generate_node,
    transform_query_node,
    should_generate,
    check_answer_quality,
    get_latest_user_query
)

LLM_MODEL = "qwen3"
BASE_URL = "http://localhost:11434"
llm = ChatOllama(
    model=LLM_MODEL, 
    base_url=BASE_URL)

In [ ]:
class RouterQuery(BaseModel):
    datasource: str = Field(description="Datasource to route query to: 'retrieve' for financial documents, 'sql_agent' for employee information, 'web_search' for general knowledge")

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    retrieved_docs: str
    rewritten_queries: List[str]

In [ ]:
def route_question(state: AgentState):
    query = get_latest_user_query(state['messages'])
    print(f"[QUERY] {query}")
    llm_router = llm.with_structured_output(RouterQuery)
    system_prompt = """You are an expert at routing user questions to the appropriate datasource.
        **Datasources:**
        1. **retrieve**: Financial documents (10-K, 10-Q, 8-K reports) about companies like Amazon, Apple, Google
        - Questions about: revenue, operating income, cash flow, segments, financial metrics, quarterly/annual performance
        - Examples: "What was Amazon's revenue in 2023?", "Compare Apple vs Google Q1 2024 revenue"
        2. **sql_agent**: Employee database with structured data
        - Questions about: employees, departments, salaries, managers, job titles, hiring dates
        - Examples: "Who is the CFO?", "List employees in Finance", "What is John's salary?"
        3. **web_search**: General knowledge outside our databases
        - Questions about: current events, general facts, topics unrelated to our data
        - Examples: "What is inflation rate?", "How does RAG work?", "Capital of France?"
        **Decision Logic:**
        - Check for financial terms (revenue, income, cash flow, quarterly, annual, 10-K, 10-Q) -> retrieve
        - Check for company names in our database (amazon, apple, google) -> retrieve
        - Check for employee-related terms (employee, department, salary, CFO, manager) -> sql_agent
        - Everything else -> web_search
        Return the datasource field only."""
    messages = [SystemMessage(system_prompt), HumanMessage(query)]
    response = llm_router.invoke(messages)
    datasource = response.datasource
    print(f"[DECISION] Routing to {datasource}")
    return datasource

route_question({'messages': [HumanMessage('what is the revenue of google?')]})


In [ ]:
def sql_agent_node(state: AgentState):
    query = get_latest_user_query(state['messages'])
    system_prompt = system_prompt = f"""You are an expert SQL analyst working with an employees database.
        Database Schema:
        {mysql_tools.SCHEMA}
        Your workflow for answering questions:
        1. Use `get_database_schema` first to understand available tables and columns (if needed)
        2. Use `generate_sql_query` to create SQL based on the question
        3. Use `execute_sql_query` to run the validated query
        4. If there's an error, use `fix_sql_error` to correct it and try again (up to 3 times)
        Rules:
        - Always follow the workflow step by step
        - If a query fails, use the fix tool and try again
        - Provide clear, informative answers
        - Be precise with table and column names
        - Handle errors gracefully and try to fix them
        - If you fail after 3 attempts, explain what went wrong
        Available tools:
        - get_database_schema: Get table structure info
        - generate_sql_query: Create SQL from question
        - execute_sql_query: Run the query
        - fix_sql_error: Fix failed queries
        Remember:
        -Always validate queries before executing them for safety.
        - Provide clear, formatted results in MARKDOWN
        - Use tables for structured data"""
    agent = create_agent(model=llm, tools=mysql_tools.ALL_SQL_TOOLS, system_prompt=system_prompt)
    result = agent.invoke({'messages': [HumanMessage(query)]})
    print(f"[SQL] Agent completed!")
    return {'messages': result['messages']}

sql_agent_node({'messages': [HumanMessage('show me 2 employees')]})

def web_search_node(state: AgentState):
    query = get_latest_user_query(state['messages'])
    system_prompt = """You are a helpful web search assistant. Be concise and accurate.
        **Workflow:**
        1. Use web_search tool to find relevant information
        2. Synthesize information from multiple sources
        3. Provide a clear, comprehensive answer
        **OUTPUT FORMAT:**
        Write a clear answer (150-250 words) in **MARKDOWN** format:
        - Use ## headings for sections
        - Use **bold** for emphasis
        - Use bullet points for key facts
        **GUIDELINES:**
        - Base your answer on web search results
        - Synthesize information from multiple sources
        - Be factual and objective"""
    agent = create_agent(model=llm, tools=[my_tools.web_search], system_prompt=system_prompt)
    result = agent.invoke({'messages': [HumanMessage(query)]})
    print(f"[WEB] Agent is completed!")
    return {'messages': result['messages']}

web_search_node({'messages': [HumanMessage('tell me about python tutorials')]})

In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver
import os
import sqlite3

db_name = "db/checkpoints.db"

os.makedirs('db', exist_ok=True)

In [ ]:
def create_adaptive_rag():
    builder = StateGraph(AgentState)
    # add nodes
    builder.add_node('retrieve', retrieve_node)
    builder.add_node('grade_documents', grade_documents_node)
    builder.add_node('generate', generate_node)
    builder.add_node('transform_query', transform_query_node)
    # add adaptive specific nodes
    builder.add_node('sql_agent', sql_agent_node)
    builder.add_node('web_search', web_search_node)
    # define edges
    # builder.add_edge(START, 'retrieve')
    builder.add_conditional_edges(START, route_question, ['retrieve', 'sql_agent', 'web_search'])
    builder.add_edge('retrieve', 'grade_documents')
    builder.add_edge('transform_query', 'retrieve')
    # conditional edges
    builder.add_conditional_edges('grade_documents', should_generate, ['transform_query', 'generate'])
    builder.add_conditional_edges('generate', check_answer_quality, ['generate', END, 'transform_query'])
    builder.add_edge('sql_agent', END)
    builder.add_edge('web_search', END)
    conn = sqlite3.connect(db_name, check_same_thread=False)
    checkpointer = SqliteSaver(conn)
    graph = builder.compile(checkpointer=checkpointer)
    return graph

agent = create_adaptive_rag()
agent

In [ ]:

config = {'configurable': {'thread_id': 'demo1'}}

query = "show me amazon's revenue in 2023?"
result = agent.invoke({'messages': [HumanMessage(query)]}, config=config)
result['messages'][-1].pretty_print()

result = agent.invoke({'messages': [HumanMessage("What is the latest AI news?")]}, config=config)
print(result['messages'][-1].content)

result = agent.invoke({'messages': [HumanMessage("How many employees are there?")]}, config=config)
print(result['messages'][-1].content)

result = agent.invoke({'messages': [HumanMessage("show me average salary department wise")]}, config=config)
print(result['messages'][-1].content)
